In [ ]:
import sys, os, time
import numpy as np
sys.path.append(os.path.abspath(".."))

from experiment import run_single_seed
from data_generators import generate_dgp
from scorers import (MahalanobisScorer, BonferroniScorer, DensityScorer,
                     KernelScorer, median_lengthscale)
from otcp_scorer import OTCPScorer
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from tqdm.notebook import tqdm

N, N_SEEDS, BASE_SEED = 10_000, 30, 21
ALPHAS = [0.1, 0.05, 0.02, 0.01]
VOL_SAMPLES = 50_000

# lengthscale = median/2 on the fit fold
class KernelMed2(KernelScorer):
    def fit(self, cal):
        self.lengthscale = median_lengthscale(cal) / cal.shape[1]
        self.auto_parameters = False
        return super().fit(cal)

class DensityMed2(DensityScorer):
    def fit(self, cal):
        self.lengthscale = median_lengthscale(cal) / cal.shape[1]
        self.auto_parameters = False
        return super().fit(cal)

def factories(seed):
    return {
        'Mahal':      lambda: MahalanobisScorer(),
        'Bonferroni': lambda: BonferroniScorer(),
        'Density':    lambda: DensityMed2(auto_parameters=False, lengthscale=1.0),
        'Kernel':     lambda: KernelMed2(auto_parameters=False, lengthscale=1.0, gamma=None),
        'OTCP':       lambda s=seed: OTCPScorer(epsilon=0.1, relative_epsilon=None,
                                                scale_mode='quantile', scale_quantile=0.99,
                                                n_target=4096, seed=s),
    }

def models():
    return {'Linear': LinearRegression(),
            'NN': MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=1000, random_state=0)}

runs, t0 = [], time.time()
for i in tqdm(range(N_SEEDS)):
    seed = BASE_SEED + i
    out = run_single_seed(generate_dgp, models(), factories(seed), ALPHAS,
                          n=N, split=(0.5, 0.25, 0.25), seed=seed,
                          volume_n_samples=VOL_SAMPLES)
    runs.append(out)
    print(f"seed {seed}  ({time.time()-t0:.0f}s)", flush=True)

In [ ]:
import pandas as pd
SC = ['Bonferroni', 'OTCP', 'Density', 'Kernel']

rows = []
for mdl in ['Linear', 'NN']:
    for sc in SC:
        for a in ALPHAS:
            r = np.array([run['results'][mdl][sc][a]['volume'] /
                          run['results'][mdl]['Mahal'][a]['volume'] for run in runs])
            c = np.array([run['results'][mdl][sc][a]['coverage'] for run in runs])
            rows.append(dict(model=mdl, scorer=sc, alpha=a, n=len(r),
                             ratio=r.mean(), ratio_sd=r.std(ddof=1),
                             cov=c.mean(), cov_sd=c.std(ddof=1)))
        cm = np.array([run['results'][mdl]['Mahal'][a]['coverage'] for run in runs])
        rows.append(dict(model=mdl, scorer='Mahal', alpha=a, n=len(cm),
                         ratio=1.0, ratio_sd=0.0, cov=cm.mean(), cov_sd=cm.std(ddof=1)))
df = pd.DataFrame(rows).round(4)
display(df.pivot_table(index=['model','scorer'], columns='alpha', values='ratio'))

# test residuals inside a region but outside the MC bounding box (should be 0)
run = runs[0]
for mdl in ['Linear', 'NN']:
    rc = run['data']['residuals_cal'][mdl]
    lo, hi = rc.min(0), rc.max(0)
    lo, hi = lo - 0.01*(hi-lo), hi + 0.01*(hi-lo)
    rt = run['data']['residuals_test'][mdl]
    outside = ((rt < lo) | (rt > hi)).any(1)
    for sc in SC:
        th = run['results'][mdl][sc][0.01]['threshold']
        s = run['fitted_scorers'][mdl][sc].score(rt[outside])
        print(f"{mdl:7s} {sc:11s} test points outside box and inside region: {(s <= th).sum()}")

In [ ]:
import matplotlib.pyplot as plt
from scatter_plotter import run_and_plot

fig_sc = {'Mahal': lambda: MahalanobisScorer(),
          'Kernel': lambda: KernelMed2(auto_parameters=False, lengthscale=1.0, gamma=None)}

fig, out = run_and_plot(generate_dgp, {'Linear': LinearRegression()}, fig_sc,
                        alphas=[0.1, 0.05, 0.02, 0.01], xlim=(-2.2, 2), ylim=(-1.8, 2),
                        seed=24, n=16_000, figsize=(6, 3.33333333))

plt.savefig('scatter_linear_SPLIT.pdf', dpi=600, bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
from scatter_plotter import run_and_plot

fig_sc = {'Mahal': lambda: MahalanobisScorer(),
          'Kernel': lambda: KernelMed2(auto_parameters=False, lengthscale=1.0, gamma=None)}
fig, out = run_and_plot(generate_dgp, {'NN':  MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=1000, random_state=0)}, fig_sc,
                        alphas=[0.1, 0.05, 0.02, 0.01], xlim=(-1.1, 1.1), ylim=(-1, 1),
                        seed=24, n=16_000, figsize=(6, 4.666666666666667), show_ylabel=False)

plt.savefig('scatter_NN_SPLIT.pdf', dpi=600, bbox_inches='tight')

In [ ]:
import json

def make_serializable(o):
    if isinstance(o, dict):  return {str(k): make_serializable(v) for k, v in o.items()}
    if isinstance(o, np.integer):  return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray):  return o.tolist()
    return o

RESULTS_DIR = 'results/synthetic_2d/'
os.makedirs(RESULTS_DIR, exist_ok=True)
for run in runs:
    payload = {
        'seed': run['seed'], 'n': N, 'alphas': ALPHAS, 
        'auto_params': make_serializable(run['auto_params']),
        'results': make_serializable(run['results']),
    }
    with open(os.path.join(RESULTS_DIR, f"seed_{run['seed']:04d}.json"), 'w') as f:
        json.dump(payload, f, indent=2)
print(f"wrote {len(runs)} files to {RESULTS_DIR}")

In [ ]:
def print_latex_synthetic(seed_paths,
                             method_order=("Bonferroni", "Mahal", "OTCP", "Density", "Kernel"),
                             models=("Linear", "NN"), label="tab:main_results_C"):
    import glob
    rows = []
    for fp in sorted(glob.glob(seed_paths)):
        seed = int(fp.split("_")[-1].split(".")[0])
        d = json.load(open(fp))
        for model, methods in d["results"].items():
            for method, alphas in methods.items():
                for alpha, m in alphas.items():
                    rows.append(dict(seed=seed, model=model, method=method,
                                     alpha=str(float(alpha))[:4], coverage=m["coverage"],
                                     volume=m["volume"], wsc=m["wsc"]))
    df = pd.DataFrame(rows)
    stats = df.groupby(["model", "method", "alpha"]).agg(["mean", "std"]).sort_index()
    means = stats.xs("mean", axis=1, level=1)[["coverage", "volume", "wsc"]]
    stds  = stats.xs("std",  axis=1, level=1)[["coverage", "volume", "wsc"]]
    best  = means.groupby(["model", "alpha"])["volume"].idxmin()

    fmt  = lambda m, s: f"{m:.5f}\\tiny{{$\\pm${s:.5f}}}"
    bold = lambda m, s: f"\\textbf{{{fmt(m, s)}}}"
    nm = len(method_order)

    lines = []
    for ai, alpha in enumerate(["0.1", "0.05", "0.02", "0.01"]):
        if ai: lines.append("    \\midrule")
        for mi, method in enumerate(method_order):
            cells = []
            for model in models:
                idx = (model, method, alpha)
                if idx not in means.index:
                    cells.extend(["---"] * 3); continue
                mc, mv, mw = means.loc[idx]; sc, sv, sw = stds.loc[idx]
                cells += [fmt(mc, sc),
                          bold(mv, sv) if best.get((model, alpha)) == idx else fmt(mv, sv),
                          fmt(mw, sw)]
            prefix = f"    \\multirow{{{nm}}}{{*}}{{{alpha}}}" if mi == 0 else "   "
            name = f"\\textbf{{{method}}}" if method == "Kernel" else method
            lines.append(f"{prefix} & {name}  & {' & '.join(cells)} \\\\")

    head = " & ".join(f"\\multicolumn{{3}}{{c}}{{{m}}}" for m in models)
    cmid = " ".join(f"\\cmidrule(lr){{{3+3*i}-{5+3*i}}}" for i in range(len(models)))
    cols = "Coverage & Volume & WSC & " * len(models)
    print(f"""\\begin{{table}}[t]
  \\centering
  \\resizebox{{\\textwidth}}{{!}}{{%
  \\begin{{tabular}}{{ll {'ccc ' * len(models)}}}
    \\toprule
    & & {head} \\\\
    {cmid}
    $\\alpha$ & Method & {cols[:-3]} \\\\
    \\midrule
{chr(10).join(lines)}
    \\bottomrule
  \\end{{tabular}}
  }}
  \\vspace{{0.5em}}
  \\caption{{Results on synthetic data across {df['seed'].nunique()} seeds. All values reported as mean $\\pm$ std. Bold indicates best volume per model and $\\alpha$.}}
  \\label{{{label}}}
\\end{{table}}""")

print_latex_synthetic(RESULTS_DIR + "seed_*.json")